<a href="https://colab.research.google.com/github/Unknown-Turtle/AMBA-AXI-Transaction-Simulator/blob/main/AMBA_AXI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
=============================================================================
AMBA AXI Triple-Master Simulator — QoS Priority Arbitration
=============================================================================
Extends the triple-master model with AXI Quality-of-Service (QoS) support.

In the previous version, all three masters shared a single FIFO queue,
meaning the CPU — the most latency-sensitive master — got stuck behind
thousands of GPU and NPU transactions. Real Arm interconnects (NIC-400,
NIC-450, CCI-550) solve this with QoS priority arbitration.

This simulator implements strict-priority arbitration with minimum
bandwidth guarantees to prevent starvation:

    CPU (Cortex-A)    GPU (Mali)      NPU (Ethos-N)
      QoS = 0          QoS = 2          QoS = 1
      HIGHEST          LOWEST           MEDIUM
         |                |                |
         v                v                v
    ─────────── AXI Interconnect ───────────
                 QoS Arbitration
       Priority 0 queue --> served first
       Priority 1 queue --> served next
       Priority 2 queue --> gets remaining BW
       (each level has a min BW guarantee)
    ────────────────────────────────────────
                      |
                      v
                  DRAM / SRAM

Arbitation modes
────────────────
  • STRICT : High priority always served first. Risk of starvation.
  • GUARANTEED : Each priority gets a minimum BW floor, then remaining
                 BW is allocated top-down by priority.  (default)

Author : Mwenya Sikazwe
Protocol: AMBA AXI (simplified behavioural model)
Usage  : python amba_axi_qos.py
Output : qos_logs.csv + comparative per-master statistics
=============================================================================
"""

import csv
import math
import random
from collections import deque
from dataclasses import dataclass
from enum import IntEnum


# ---------------------------------------------------------------------------
# 1. QoS Priority Levels
# ---------------------------------------------------------------------------

class QoS(IntEnum):
    """
    AXI QoS priority levels.  Lower number = higher priority.
    In real AXI, the AxQOS signal is a 4-bit field (0-15).
    We use three levels for clarity.
    """
    HIGH   = 0   # CPU — latency-critical OS/app workloads
    MEDIUM = 1   # NPU — real-time inference deadlines
    LOW    = 2   # GPU — latency-tolerant, bandwidth-hungry


# ---------------------------------------------------------------------------
# 2. Transaction (now with QoS)
# ---------------------------------------------------------------------------

@dataclass
class Transaction:
    """
    A single AXI transaction with a QoS priority tag.

    In real AXI, the AxQOS field travels alongside the address on
    the AR/AW channels and tells the interconnect how urgently
    this transaction should be forwarded.
    """
    tx_id: int
    source: str
    tx_type: str
    priority: int           # QoS level (0 = highest)
    start_cycle: int
    end_cycle: int = -1
    queue_size: int = 0

    @property
    def latency(self) -> int:
        return self.end_cycle - self.start_cycle if self.end_cycle >= 0 else -1


# ---------------------------------------------------------------------------
# 3. CPU Master — QoS HIGH (priority 0)
# ---------------------------------------------------------------------------

class CPU_Master:
    """
    CPU with highest QoS priority — its transactions jump to the
    front of the arbitration order. This reflects real SoC design
    where OS responsiveness must not be affected by GPU/NPU load.
    """

    def __init__(self, base_injection_rate: float = 3.0, read_ratio: float = 0.7):
        self.base_injection_rate = base_injection_rate
        self.read_ratio = read_ratio
        self.name = 'CPU'
        self.priority = QoS.HIGH

    def generate(self, cycle: int, queue_size: int, next_id: int) -> list:
        count = self._poisson_sample()
        txns = []
        for i in range(count):
            tx_type = 'READ' if random.random() < self.read_ratio else 'WRITE'
            txns.append(Transaction(
                tx_id=next_id + i,
                source=self.name,
                tx_type=tx_type,
                priority=self.priority,
                start_cycle=cycle,
                queue_size=queue_size + i,
            ))
        return txns

    def _poisson_sample(self) -> int:
        L = math.exp(-self.base_injection_rate)
        k, p = 0, 1.0
        while True:
            k += 1
            p *= random.random()
            if p < L:
                return k - 1


# ---------------------------------------------------------------------------
# 4. GPU Master — QoS LOW (priority 2)
# ---------------------------------------------------------------------------

class GPU_Master:
    """
    GPU with lowest QoS priority — it gets whatever bandwidth
    remains after CPU and NPU are served.  GPUs are latency-tolerant
    because they mask memory delays by switching between thousands
    of shader threads.
    """

    def __init__(
        self,
        burst_rate: int = 12,
        active_cycles: int = 400,
        window_cycles: int = 500,
        read_ratio: float = 0.85,
    ):
        self.burst_rate = burst_rate
        self.active_cycles = active_cycles
        self.window_cycles = window_cycles
        self.read_ratio = read_ratio
        self.name = 'GPU'
        self.priority = QoS.LOW

    def generate(self, cycle: int, queue_size: int, next_id: int) -> list:
        phase = cycle % self.window_cycles
        if phase >= self.active_cycles:
            return []

        count = self.burst_rate + random.randint(-2, 2)
        count = max(0, count)

        txns = []
        for i in range(count):
            tx_type = 'READ' if random.random() < self.read_ratio else 'WRITE'
            txns.append(Transaction(
                tx_id=next_id + i,
                source=self.name,
                tx_type=tx_type,
                priority=self.priority,
                start_cycle=cycle,
                queue_size=queue_size + i,
            ))
        return txns


# ---------------------------------------------------------------------------
# 5. NPU Master — QoS MEDIUM (priority 1)
# ---------------------------------------------------------------------------

class NPU_Master:
    """
    NPU with medium QoS priority.  NPUs have real-time inference
    deadlines (e.g. 16ms per frame for camera preview) so they
    get priority over the GPU but yield to the CPU.
    """

    def __init__(
        self,
        load_rate: int = 15,
        store_rate: int = 8,
        load_cycles: int = 150,
        compute_cycles: int = 200,
        store_cycles: int = 80,
        idle_cycles: int = 70,
    ):
        self.load_rate = load_rate
        self.store_rate = store_rate
        self.load_cycles = load_cycles
        self.compute_cycles = compute_cycles
        self.store_cycles = store_cycles
        self.idle_cycles = idle_cycles
        self.name = 'NPU'
        self.priority = QoS.MEDIUM
        self.layer_period = (
            load_cycles + compute_cycles + store_cycles + idle_cycles
        )

    def generate(self, cycle: int, queue_size: int, next_id: int) -> list:
        phase_offset = cycle % self.layer_period

        if phase_offset < self.load_cycles:
            count = self.load_rate + random.randint(-2, 2)
            count = max(0, count)
            tx_type = 'READ'
        elif phase_offset < self.load_cycles + self.compute_cycles:
            return []
        elif phase_offset < self.load_cycles + self.compute_cycles + self.store_cycles:
            count = self.store_rate + random.randint(-1, 1)
            count = max(0, count)
            tx_type = 'WRITE'
        else:
            return []

        txns = []
        for i in range(count):
            txns.append(Transaction(
                tx_id=next_id + i,
                source=self.name,
                tx_type=tx_type,
                priority=self.priority,
                start_cycle=cycle,
                queue_size=queue_size + i,
            ))
        return txns


# ---------------------------------------------------------------------------
# 6. AXI Bus — QoS-Aware Arbitration
# ---------------------------------------------------------------------------

class AXI_Bus_QoS:
    """
    AXI interconnect with QoS-aware priority arbitration.

    Instead of a single FIFO, the bus maintains separate queues for
    each priority level.  Each cycle, bandwidth is allocated as:

        GUARANTEED mode (default):
        ─────────────────────────
        1. Each priority level drains up to its `min_bw` guarantee.
        2. Any remaining bandwidth is allocated top-down by priority:
           HIGH gets leftover first, then MEDIUM, then LOW.

        STRICT mode:
        ────────────
        1. HIGH queue is fully drained (up to total BW).
        2. MEDIUM gets remaining BW.
        3. LOW gets whatever is left (may starve).

    Parameters
    ----------
    bandwidth       : Total bus bandwidth (txns/cycle).
    min_bw          : Dict mapping QoS level → minimum guaranteed BW.
                      Only used in GUARANTEED mode.
    mode            : 'GUARANTEED' or 'STRICT'.
    max_queue_depth : Per-priority hard queue limit (0 = unlimited).
    """

    def __init__(
        self,
        bandwidth: int = 10,
        min_bw: dict | None = None,
        mode: str = 'GUARANTEED',
        max_queue_depth: int = 0,
    ):
        self.bandwidth = bandwidth
        self.mode = mode.upper()
        self.max_queue_depth = max_queue_depth
        self.dropped_count = 0

        # Default minimum bandwidth guarantees
        self.min_bw = min_bw or {
            QoS.HIGH:   3,   # CPU always gets at least 3
            QoS.MEDIUM: 2,   # NPU always gets at least 2
            QoS.LOW:    1,   # GPU always gets at least 1
        }
        # Remaining 4 slots allocated by priority (10 - 3 - 2 - 1 = 4)

        # Separate queue per priority level
        self.queues: dict[int, deque] = {
            QoS.HIGH:   deque(),
            QoS.MEDIUM: deque(),
            QoS.LOW:    deque(),
        }

    @property
    def queue_size(self) -> int:
        return sum(len(q) for q in self.queues.values())

    def queue_size_by_priority(self, priority: int) -> int:
        return len(self.queues[priority])

    def enqueue(self, transactions: list) -> None:
        """Route each transaction to its priority queue."""
        for tx in transactions:
            q = self.queues[tx.priority]
            if self.max_queue_depth and len(q) >= self.max_queue_depth:
                self.dropped_count += 1
                continue
            q.append(tx)

    def dequeue(self) -> list:
        """
        Drain transactions using the configured arbitration mode.
        """
        if self.mode == 'STRICT':
            return self._dequeue_strict()
        else:
            return self._dequeue_guaranteed()

    def _dequeue_strict(self) -> list:
        """
        Strict priority: serve HIGH fully, then MEDIUM, then LOW.
        Lower priorities risk starvation under heavy load.
        """
        forwarded = []
        remaining_bw = self.bandwidth

        for priority in [QoS.HIGH, QoS.MEDIUM, QoS.LOW]:
            q = self.queues[priority]
            while remaining_bw > 0 and q:
                forwarded.append(q.popleft())
                remaining_bw -= 1

        return forwarded

    def _dequeue_guaranteed(self) -> list:
        """
        Guaranteed mode:
          Phase 1 — Each priority drains up to its min_bw guarantee.
          Phase 2 — Remaining BW allocated by priority (high → low).

        This prevents starvation while still favouring high-priority
        traffic.  Mirrors real Arm NIC QoS regulators.
        """
        forwarded = []
        remaining_bw = self.bandwidth

        # Phase 1: Guaranteed minimums
        for priority in [QoS.HIGH, QoS.MEDIUM, QoS.LOW]:
            q = self.queues[priority]
            guarantee = self.min_bw.get(priority, 0)
            served = 0
            while served < guarantee and remaining_bw > 0 and q:
                forwarded.append(q.popleft())
                served += 1
                remaining_bw -= 1

        # Phase 2: Remaining BW goes to highest priority first
        for priority in [QoS.HIGH, QoS.MEDIUM, QoS.LOW]:
            q = self.queues[priority]
            while remaining_bw > 0 and q:
                forwarded.append(q.popleft())
                remaining_bw -= 1

        return forwarded


# ---------------------------------------------------------------------------
# 7. Memory Slave
# ---------------------------------------------------------------------------

class Memory_Slave:
    """Completes transactions and records them."""

    def __init__(self):
        self.completed: list[Transaction] = []

    def process(self, transactions: list, cycle: int) -> list:
        for tx in transactions:
            tx.end_cycle = cycle
            self.completed.append(tx)
        return transactions


# ---------------------------------------------------------------------------
# 8. Simulation Engine
# ---------------------------------------------------------------------------

def run_qos_simulation(
    total_cycles: int = 10_000,
    # -- Bus --
    bus_bandwidth: int = 10,
    qos_mode: str = 'GUARANTEED',
    min_bw_high: int = 3,
    min_bw_medium: int = 2,
    min_bw_low: int = 1,
    max_queue_depth: int = 0,
    # -- CPU --
    cpu_injection_rate: float = 3.0,
    cpu_read_ratio: float = 0.7,
    # -- GPU --
    gpu_burst_rate: int = 12,
    gpu_active_cycles: int = 400,
    gpu_window_cycles: int = 500,
    gpu_read_ratio: float = 0.85,
    # -- NPU --
    npu_load_rate: int = 15,
    npu_store_rate: int = 8,
    npu_load_cycles: int = 150,
    npu_compute_cycles: int = 200,
    npu_store_cycles: int = 80,
    npu_idle_cycles: int = 70,
    # -- Output --
    csv_filename: str = 'qos_logs.csv',
    seed: int | None = 42,
) -> None:
    """
    Run the QoS-enabled triple-master simulation.

    Compare the results with the non-QoS triple_master_logs.csv
    to see how priority arbitration dramatically reduces CPU latency.
    """

    if seed is not None:
        random.seed(seed)

    # -- Components --------------------------------------------------------
    cpu = CPU_Master(base_injection_rate=cpu_injection_rate,
                     read_ratio=cpu_read_ratio)
    gpu = GPU_Master(burst_rate=gpu_burst_rate,
                     active_cycles=gpu_active_cycles,
                     window_cycles=gpu_window_cycles,
                     read_ratio=gpu_read_ratio)
    npu = NPU_Master(load_rate=npu_load_rate,
                     store_rate=npu_store_rate,
                     load_cycles=npu_load_cycles,
                     compute_cycles=npu_compute_cycles,
                     store_cycles=npu_store_cycles,
                     idle_cycles=npu_idle_cycles)

    min_bw = {QoS.HIGH: min_bw_high, QoS.MEDIUM: min_bw_medium, QoS.LOW: min_bw_low}
    bus = AXI_Bus_QoS(bandwidth=bus_bandwidth, min_bw=min_bw,
                      mode=qos_mode, max_queue_depth=max_queue_depth)
    slave = Memory_Slave()

    next_tx_id = 0
    max_queue_observed = 0
    queue_history = []

    # -- Header ------------------------------------------------------------
    npu_period = npu.layer_period
    guaranteed_sum = sum(min_bw.values())
    flexible_bw = bus_bandwidth - guaranteed_sum

    print(f"{'='*65}")
    print(f"  AMBA AXI QoS Simulator – {total_cycles:,} cycles")
    print(f"  Arbitration mode : {qos_mode}")
    print(f"  Bus bandwidth    : {bus_bandwidth} txns/cycle")
    if qos_mode == 'GUARANTEED':
        print(f"    ├─ CPU  min BW  : {min_bw_high} (QoS HIGH)")
        print(f"    ├─ NPU  min BW  : {min_bw_medium} (QoS MEDIUM)")
        print(f"    ├─ GPU  min BW  : {min_bw_low} (QoS LOW)")
        print(f"    └─ Flexible pool: {flexible_bw} (priority-first)")
    print(f"  CPU injection (λ): {cpu_injection_rate}")
    print(f"  GPU burst rate   : {gpu_burst_rate} txns/cycle "
          f"({gpu_active_cycles}/{gpu_window_cycles} active)")
    print(f"  NPU layer period : {npu_period} cycles "
          f"(load {npu_load_cycles} → compute {npu_compute_cycles} "
          f"→ store {npu_store_cycles} → idle {npu_idle_cycles})")
    print(f"{'='*65}\n")

    # ======================================================================
    #  CLOCK LOOP
    # ======================================================================
    for cycle in range(total_cycles):

        q = bus.queue_size

        # 1) CPU generates transactions (QoS HIGH)
        cpu_txns = cpu.generate(cycle, q, next_tx_id)
        next_tx_id += len(cpu_txns)

        # 2) GPU generates transactions (QoS LOW)
        gpu_txns = gpu.generate(cycle, q, next_tx_id)
        next_tx_id += len(gpu_txns)

        # 3) NPU generates transactions (QoS MEDIUM)
        npu_txns = npu.generate(cycle, q, next_tx_id)
        next_tx_id += len(npu_txns)

        # 4) All enter priority-separated queues
        bus.enqueue(cpu_txns + gpu_txns + npu_txns)

        # 5) QoS arbitration drains by priority
        forwarded = bus.dequeue()
        slave.process(forwarded, cycle)

        # 6) Track queue depth
        q = bus.queue_size
        queue_history.append(q)
        max_queue_observed = max(max_queue_observed, q)

        # 7) Progress
        if (cycle + 1) % 2000 == 0 or cycle == 0:
            q_hi = bus.queue_size_by_priority(QoS.HIGH)
            q_md = bus.queue_size_by_priority(QoS.MEDIUM)
            q_lo = bus.queue_size_by_priority(QoS.LOW)
            print(f"  Cycle {cycle+1:>6,} / {total_cycles:,}  |  "
                  f"Queue: {q:>7,}  "
                  f"(H:{q_hi:>5,} M:{q_md:>5,} L:{q_lo:>5,})  |  "
                  f"Done: {len(slave.completed):>9,}")

    # ======================================================================
    #  EXPORT CSV
    # ======================================================================
    with open(csv_filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            'Transaction_ID', 'Source', 'Type', 'QoS_Priority',
            'Start_Cycle', 'End_Cycle', 'Latency_Cycles',
            'Queue_Size_At_Creation',
        ])
        for tx in slave.completed:
            writer.writerow([
                tx.tx_id, tx.source, tx.tx_type,
                QoS(tx.priority).name, tx.start_cycle,
                tx.end_cycle, tx.latency, tx.queue_size,
            ])

    # ======================================================================
    #  STATISTICS
    # ======================================================================
    all_txns = slave.completed
    cpu_done = [tx for tx in all_txns if tx.source == 'CPU']
    gpu_done = [tx for tx in all_txns if tx.source == 'GPU']
    npu_done = [tx for tx in all_txns if tx.source == 'NPU']

    def stats(txns, label, qos_label):
        if not txns:
            print(f"  {label}: No completed transactions")
            return
        lats = [tx.latency for tx in txns]
        reads = sum(1 for tx in txns if tx.tx_type == 'READ')
        avg_lat = sum(lats) / len(lats)
        print(f"  {label}  [QoS {qos_label}]")
        print(f"    Completed      : {len(txns):>10,}")
        print(f"      READs        : {reads:>10,}")
        print(f"      WRITEs       : {len(txns)-reads:>10,}")
        print(f"    Avg latency    : {avg_lat:>10.2f} cycles")
        print(f"    Min latency    : {min(lats):>10,} cycles")
        print(f"    Max latency    : {max(lats):>10,} cycles")

    avg_queue = sum(queue_history) / len(queue_history) if queue_history else 0

    print(f"\n{'='*65}")
    print(f"  SIMULATION COMPLETE  ({qos_mode} mode)")
    print(f"{'='*65}")
    stats(cpu_done, "CPU (Cortex-A)", "HIGH")
    print(f"  {'─'*55}")
    stats(npu_done, "NPU (Ethos-N) ", "MEDIUM")
    print(f"  {'─'*55}")
    stats(gpu_done, "GPU (Mali)     ", "LOW")
    print(f"  {'─'*55}")
    print(f"  SYSTEM TOTALS")
    print(f"    Total completed: {len(all_txns):>10,}")
    print(f"    Dropped        : {bus.dropped_count:>10,}")
    print(f"    Pending (end)  : {bus.queue_size:>10,}")
    print(f"    Avg queue depth: {avg_queue:>10.2f}")
    print(f"    Peak queue     : {max_queue_observed:>10,}")
    print(f"    CSV            : {csv_filename}")
    print(f"{'='*65}\n")

    # ── Quick comparison hint ──
    print("  💡 Compare with triple_master_logs.csv (no QoS) to see the")
    print("     impact of priority arbitration on CPU latency.\n")


# ---------------------------------------------------------------------------
# 9. Entry Point
# ---------------------------------------------------------------------------

if __name__ == '__main__':
    run_qos_simulation(
        total_cycles=10_000,
        # -- Bus with QoS --
        bus_bandwidth=10,
        qos_mode='GUARANTEED',       # Try 'STRICT' for comparison
        min_bw_high=3,               # CPU guaranteed 3 txns/cycle
        min_bw_medium=2,             # NPU guaranteed 2 txns/cycle
        min_bw_low=1,                # GPU guaranteed 1 txn/cycle
        # -- CPU --
        cpu_injection_rate=3.0,
        cpu_read_ratio=0.7,
        # -- GPU --
        gpu_burst_rate=12,
        gpu_active_cycles=400,
        gpu_window_cycles=500,
        gpu_read_ratio=0.85,
        # -- NPU --
        npu_load_rate=15,
        npu_store_rate=8,
        npu_load_cycles=150,
        npu_compute_cycles=200,
        npu_store_cycles=80,
        npu_idle_cycles=70,
        # -- Output --
        csv_filename='qos_logs.csv',
        seed=42,
    )


  AMBA AXI QoS Simulator – 10,000 cycles
  Arbitration mode : GUARANTEED
  Bus bandwidth    : 10 txns/cycle
    ├─ CPU  min BW  : 3 (QoS HIGH)
    ├─ NPU  min BW  : 2 (QoS MEDIUM)
    ├─ GPU  min BW  : 1 (QoS LOW)
    └─ Flexible pool: 4 (priority-first)
  CPU injection (λ): 3.0
  GPU burst rate   : 12 txns/cycle (400/500 active)
  NPU layer period : 500 cycles (load 150 → compute 200 → store 80 → idle 70)

  Cycle      1 / 10,000  |  Queue:      19  (H:    0 M:    9 L:   10)  |  Done:        10
  Cycle  2,000 / 10,000  |  Queue:  16,867  (H:    0 M:    0 L:16,867)  |  Done:    20,000
  Cycle  4,000 / 10,000  |  Queue:  33,612  (H:    0 M:    0 L:33,612)  |  Done:    40,000
  Cycle  6,000 / 10,000  |  Queue:  50,193  (H:    0 M:    0 L:50,193)  |  Done:    60,000
  Cycle  8,000 / 10,000  |  Queue:  66,922  (H:    0 M:    0 L:66,922)  |  Done:    80,000
  Cycle 10,000 / 10,000  |  Queue:  83,706  (H:    0 M:    0 L:83,706)  |  Done:   100,000

  SIMULATION COMPLETE  (GUARANTEED mode)
  